# 5.4 Description of the block power method for the leading transfer spectrum

In a standard setting, a power method finds the dominant eigenvector of a transfer matrix $E$ by repeatedly applying it to a random initial state. The convergence rate is strictly governed by the spectral gap ratio $|\mu_1/\mu_0|$: each application of $E$ suppresses the subleading states by this exact factor.

However, quenching to a critical point physically drives this gap ratio to exactly one—a phenomenon known as **emergent dual unitarity**. The better the conformal physics, the worse the power method performs. As the temporal extent $T$ grows and the interesting scaling sets in, the leading eigenvalues become degenerate. The single-vector iteration slows down critically and eventually stalls, unable to distinguish the dominant eigenvector from its near-degenerate partners.

To survive this physical bottleneck, we must abandon the single-vector approach. Instead, we iterate a *block* of $k$ vectors simultaneously, allowing them to span the entire quasi-degenerate cluster. The convergence of the leading pair is then governed not by $|\mu_1/\mu_0|$, but by $|\mu_k/\mu_0|$ (the gap between our block and the rest of the spectrum). This modified gap remains comfortably finite as long as our block is large enough to contain the entire degenerate cluster.

Because our transfer matrix $E$ is non-Hermitian and non-normal, this approach requires us to work strictly in a biorthogonal, bilinear language. This section breaks down that mathematical reformulation and the software engineering behind it, as implemented in `ITransverse.jl` (specifically `block_transfer_eigs` in `transverse_tools.jl`).

## The two-sided block and the oblique projection

We cannot rely on standard orthogonal bases. Instead, we initialize $k$ right vectors $|R_1\rangle,\dots,|R_k\rangle$ and $k$ independent left vectors $\langle L_1|,\dots,\langle L_k|$. These are temporal MPS living on the same time sites, seeded with random complex tensors.

A single iteration applies $E$ to every member of the right block, and its transpose to every member of the left block:

$$|\tilde R_j\rangle = E\,|R_j\rangle, \qquad
  \langle \tilde L_j| = \langle L_j|\,E
  \quad\Longleftrightarrow\quad
  |\tilde L_j\rangle = E^{\mathsf T}|L_j\rangle.
  \tag{B1}
$$

*(Note: The transpose here is the adjoint with respect to the bilinear form, not the regular Hermitian adjoint).*

Just as in the single-vector method, each MPO application physically inflates the bond dimension. Therefore, every resulting tensor network in (B1) must be compressed back to a maximum capacity $\chi_{\max}$ (detailed in the truncation modes below) and renormalized.

## The projected pencil: From tensor networks to a subspace eigenproblem

The block method bypasses the failure of the transfer matrix filter by shifting to a subspace approach. We assume that the true dominant right eigenvectors $|\psi\rangle$ of $E$ live within the linear span of our iterated right block. Mathematically, we approximate the exact eigenvector as a linear combination of our $k$ block vectors:
$$|\psi\rangle \approx \sum_{j=1}^k v_j |R_j\rangle$$
where $v_j$ are simple scalar coefficients. We demand that this state satisfies the eigenvalue equation $E |\psi\rangle = \theta |\psi\rangle$. Because our truncated space cannot capture the exact state perfectly, plugging in our approximation leaves a residual error:
$$\sum_{j=1}^k v_j E |R_j\rangle - \theta \sum_{j=1}^k v_j |R_j\rangle \neq 0$$

To solve for the coefficients $v_j$, we enforce that this error vanishes when viewed from the dual space spanned by our left block. By multiplying the entire equation from the left by each $\langle L_i|$, we perform an oblique (Petrov-Galerkin) projection:
$$\sum_{j=1}^k \langle L_i | E | R_j \rangle v_j = \theta \sum_{j=1}^k \langle L_i | R_j \rangle v_j$$

This simple projection is the core of the method: it compresses the massive, computationally intractable tensor network operations into two small $k\times k$ matrices:
$$S_{ij} = \langle L_i | R_j\rangle, \qquad M_{ij} = \langle L_i | E | R_j \rangle \tag{B2}$$
both evaluated with the bilinear overlap (no conjugation). 
- $S$ is the overlap metric of the projection, measuring how the left and right spaces are angled against each other.
- $M$ is the transfer matrix $E$ projected into this small subspace. 

Substituting these matrices back into our projected equation reduces the massive tensor problem into a standard generalized eigenproblem for the Ritz values $\theta$:
$$M\,v = \theta\, S\, v \tag{B3}$$

A critical subtlety arises here. Near a degeneracy (emergent dual unitarity), the repeated application of $E$ pulls the columns of the right block toward the same physical state. As the vectors become increasingly parallel, they become linearly dependent, making the metric $S$ nearly singular. Consequently, standard solvers like `eigen(M, S)` attempt to divide by near-zero, returning infinite or wildly scattered eigenvalues. 

To safely extract the spectrum, we instead use the Moore-Penrose pseudoinverse with a relative tolerance (e.g., $10^{-12}$):
$$W = S^{+} M, \qquad W = V\,\Theta\,V^{-1} \tag{B4}$$
By using $S^{+}$, directions in the block that have effectively collapsed onto each other are safely projected out rather than inverted. The extracted eigenvalues $\Theta = \mathrm{diag}(\theta_1,\dots,\theta_k)$, sorted by modulus, provide our stable estimates for $\mu_0,\dots,\mu_{k-1}$. The columns of $V$ dictate exactly which linear combination of the current right block ($v_j$) best constructs the true dominant eigenvectors.

## Pairing the left coefficients: A zero-risk strategy

Now that we have the right-side mixing coefficients ($v_j$) and the eigenvalues ($\theta_j$), we need the corresponding left-side coefficients ($u_j$) to mix our left block $\langle L_1|, \dots, \langle L_k|$. 

The obvious approach is to solve the transposed eigenproblem, $u^{\mathsf T} M = \theta u^{\mathsf T} S$, with a second, independent solver call. But due to numerical instabilities near degeneracy, the method might pair a left vector with the wrong right eigenvalue. To avoid this, we derive the left coefficients directly from the right-side solution. 

If we define $y^{\mathsf T} = u^{\mathsf T} S$, the transposed problem becomes $y^{\mathsf T} W = \theta\, y^{\mathsf T}$. This means the vectors $y$ are simply the left eigenvectors of our previously calculated matrix $W$, which are exactly the rows of $V^{-1}$. Therefore, we can construct the left coefficients as:

$$u_j = \big(S^{+}\big)^{\mathsf T} \big(V^{-1}\big)^{\mathsf T} e_j \tag{B5}$$

This formulation guarantees that $u_j$ is perfectly paired with $\theta_j$ by construction, eliminating the need for heuristic matching and automatically enforcing biorthogonality.

## De-mixing and surviving parallel collapse

With the coefficients $V$ and $U$ in hand, we update our blocks by mixing the applied vectors:

$$|R_j\rangle \leftarrow \sum_i V_{ij}\, |\tilde R_i\rangle, \qquad
  \langle L_j| \leftarrow \sum_i U_{ij}\, \langle \tilde L_i| \tag{B6}$$

Because adding massive tensor networks causes the bond dimension to explode, these newly mixed states must be compressed back to $\chi_{\max}$. This is done in one of two modes:
* **RTM mode:** Compresses the matched pair $(L_j, R_j)$ *jointly* based on the transfer matrix they form. This is physically optimal and computationally cheap, but relies on non-Hermitian mathematics that can become unstable exactly at the degeneracy.
* **RDM mode:** Compresses each vector *independently* using its own Hermitian density matrix. This is more expensive and ignores the left-right pairing, but remains numerically stable through the cluster, making it our safe fallback.

Even with perfect compression, Equation (B6) has a structural flaw: it forces the block into the *eigenvector* basis. As emergent dual unitarity pulls the top eigenvalues together, their corresponding eigenvectors become nearly parallel. Rotating massive, noisy tensor networks onto nearly parallel directions acts like an amplifier for that noise, blowing up the condition number of the matrix $V$.

To fix this, we abandon the attempt to track individual eigenvectors inside the degenerate cluster. Instead, we only track the well-conditioned *subspace* they span. We do this by QR-orthonormalizing the coefficient matrices before applying them:

$$V \to Q_V, \qquad U \to Q_U, \qquad (Q^\dagger Q = \mathbb 1) \tag{B7}$$

The Ritz values are still calculated exactly, but the physical tensor networks we carry between iterations now form a stable, orthogonal basis rather than a fragile, near-parallel one.

## Convergence and the true physical gap

We declare convergence when the Ritz values $\theta$ stop moving between iterations. However, we must account for two physical quirks of the non-Hermitian transfer spectrum:

1. **The $\pm$ Partner:** The spectrum comes in pairs. For every dominant $\mu_0$, there is a partner with nearly the same modulus but a flipped sign ($\approx -\mu_0$). Because we sort by modulus, $\mu_0$ and its partner frequently swap positions between iterations. We must track them by continuity in the complex plane to avoid registering a spurious "jump" in convergence.
2. **The True Gap:** Because of this partner, the simple ratio $|\theta_2|/|\theta_1|$ does not measure the true spectral gap; it only measures the tiny splitting between $\mu_0$ and $-\mu_0$. The meaningful physical gap compares the dominant state to the highest state that is *neither* $\mu_0$ nor its negative partner:

$$\mathrm{gap}(T) = \frac{\max\{|\theta_i| : i \neq i_0,\, i \neq i_{\text{partner}}\}}{|\theta_{i_0}|} \tag{B8}$$

*Practical Note: To accelerate the solver across different times $T$, we use a "warm-start" strategy. The converged block from $T$ is padded with a small random tail and used as the initial guess for $T+dt$. Furthermore, early iterations use loose compression tolerances, tightening only as the block approaches convergence.*

## The ultimate limit of the method

While this block method pushed our analysis of the non-integrable Alcaraz model out to $T \approx 9$, it cannot permanently bypass the entanglement barrier. It is crucial to separate what numerical tricks can fix from what physics forbids.

The block method with exact pairing and QR-orthogonalization gives us robust access to the spectrum (the eigenvalues and the gap) arbitrarily deep into the dual-unitary regime. However, extracting physical observables like temporal entanglement entropy requires the individual eigenvector. As emergent dual unitarity closes the physical gap, the condition number of that individual eigenvector scales as $\sim 1/\text{gap}$. 

Any numerical representation of the dominant eigenvector degrades at a rate dictated by the physics itself, not the iteration scheme. Therefore:

$$\underbrace{\text{spectrum of } E}_{\text{robust: eigenvalue conditioning}}
  \qquad\text{vs}\qquad
  \underbrace{S^{\mathrm{gen}}_2 \text{ from } (L_0, R_0)}_{\text{bounded by } 1/\text{gap}} \tag{B9}$$

The entropy profile is only trustworthy in the pre-wall window where the leading pair remains isolated. When a model allows for a left-right symmetric MPO (like the Ising or our specific $XXZ$ decomposition), the non-Hermitian complications vanish entirely, allowing us to directly measure how much of the early wall was caused by physics versus the asymmetry of the method itself.